# 1-4. LLM API와 추론

⏱ **소요시간**: 15분

## 학습 목표
- `common.get_llm()` 헬퍼로 제공자 독립적인 LLM 호출을 작성한다.
- 스트리밍과 temperature / top_p 차이를 관찰한다.
- Tool calling (function calling) 패턴을 간단한 예제로 검증한다.
- 컨텍스트 윈도우 사용량과 비용을 계산한다 (gpt-4o-mini 기준).

## 핵심 키워드
`ChatOpenAI`, `streaming`, `temperature`, `top_p`, `tool calling`, `context window`, `pricing`

## 0. 준비

`common.llm.get_llm()`은 `.env`의 `LLM_PROVIDER` 값에 따라 `ChatOpenAI` 또는 `ChatOllama`를 돌려준다. 이 노트북은 기본 설정인 OpenAI 기준으로 작성되었지만, Day 2 S4-5에서 `.env`만 바꾸면 동일하게 Ollama로 실행된다.

미리 `.env` 파일에 `OPENAI_API_KEY`가 설정되어 있어야 한다.

In [ ]:
import sys
sys.path.insert(0, '../..')

from common import get_llm, provider_badge

print(provider_badge())

## 1. 기본 호출

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

llm = get_llm(temperature=0.0)

messages = [
    SystemMessage(content='다음 질문에 한국어로 2문장 이내로 답하세요.'),
    HumanMessage(content='금융권에서 망분리 규제가 필요한 이유는?'),
]

response = llm.invoke(messages)
print(response.content)
print()
print('usage:', response.response_metadata.get('token_usage'))

## 2. 스트리밍

UI 체감상 중요. `get_llm(streaming=True)` + `.stream(messages)` 으로 토큰 단위 생성을 받는다.

In [ ]:
llm_stream = get_llm(temperature=0.0, streaming=True)

messages = [
    SystemMessage(content='한국어로 간결하게 설명합니다.'),
    HumanMessage(content='전자금융거래법의 자금이체 책임 조항을 세 줄로 설명해주세요.'),
]

print('[streaming]')
for chunk in llm_stream.stream(messages):
    print(chunk.content, end='', flush=True)
print('\n[done]')

## 3. temperature · top_p 비교

- **temperature** (0~2): logit을 $\frac{1}{T}$로 나눐 softmax. 0에 가까울수록 결정적, 1이 기본, 1.5+ 는 상상력.
- **top_p** (0~1): 누적 확률이 p 이상이 되는 최소의 토큰 집합에서만 샘플링 (nucleus sampling).
- 통상 **둘 중 하나만** 조절한다.

In [ ]:
prompt = HumanMessage(content='은행원이 고객에게 신용카드를 권하는 시나리오를 한 문장으로 작성하세요.')

for t in [0.0, 0.7, 1.5]:
    llm_t = get_llm(temperature=t)
    print(f'--- temperature={t} ---')
    for trial in range(2):
        print(f'  [trial {trial + 1}]', llm_t.invoke([prompt]).content.strip())
    print()

temperature=0에서는 두 시도가 거의 같아야 하고, 1.5에서는 상당히 다르게 나와야 한다 (OpenAI는 seed 입력해도 완전히 결정적이지는 않음).

## 4. Tool Calling (함수 호출)

LLM이 직접 외부 도구를 호출하도록 일안하고, 실행은 우리 코드가 한다. `bind_tools` 를 쓰면 LangChain이 연동해준다.

In [ ]:
from langchain_core.tools import tool


@tool
def get_exchange_rate(base: str, target: str) -> float:
    """두 통화 간 환율을 리턴한다. 실제 구현에서는 외부 API 호출.

    Args:
        base: 기준 통화 코드 (예: 'USD')
        target: 대상 통화 코드 (예: 'KRW')
    """
    # 데모용 고정 매핑
    table = {('USD', 'KRW'): 1380.5, ('EUR', 'KRW'): 1490.2, ('JPY', 'KRW'): 9.1}
    return table.get((base.upper(), target.upper()), 0.0)


@tool
def interest_amount(principal: float, annual_rate_pct: float, days: int) -> float:
    """원금, 연이자율(%), 기간(일) 기반으로 단리 이자를 계산한다."""
    return principal * annual_rate_pct / 100 * days / 365


tools = [get_exchange_rate, interest_amount]
llm_with_tools = get_llm(temperature=0).bind_tools(tools)

user_q = '1000만원을 연 3.5%로 90일 예치하면 이자는 얼마인가요? 그리고 그 이자를 달러로 환산하면?'
ai_msg = llm_with_tools.invoke([HumanMessage(content=user_q)])

print('모델이 요청한 tool call:')
for call in ai_msg.tool_calls:
    print('  -', call['name'], call['args'])

In [ ]:
# 도구를 실제 실행해 결과를 다시 모델에 넘긴다 (증거 기반 최종 답변 생성)
from langchain_core.messages import ToolMessage

tool_lookup = {t.name: t for t in tools}
history = [HumanMessage(content=user_q), ai_msg]

for call in ai_msg.tool_calls:
    result = tool_lookup[call['name']].invoke(call['args'])
    history.append(ToolMessage(content=str(result), tool_call_id=call['id']))

final = llm_with_tools.invoke(history)
print(final.content)

## 5. 컨텍스트 윈도우·비용 계산

gpt-4o-mini (2024-11 기준):

| 항목 | 값 |
|---|---|
| Context window | 128K tokens |
| Max output | 16K tokens |
| Input 단가 | $0.150 / 1M tokens |
| Cached input | $0.075 / 1M tokens |
| Output 단가 | $0.600 / 1M tokens |

실제 시나리오 계산.

In [ ]:
import tiktoken
enc = tiktoken.get_encoding('o200k_base')

system_prompt = '당신은 한국 금융 규제 전문가입니다. 법령 조문을 인용해 정확하게 답변하세요.'
retrieved_docs_chars = 8000   # RAG에서 불러온 문맥 길이 (문자 수)
user_question = '망분리 권고안에 따른 클라우드 예외 조건에 대해 설명하세요.'
# 한국어 기준 o200k는 chars/tok ≈ 1.4
doc_tokens = int(retrieved_docs_chars / 1.4)

input_tokens = (
    len(enc.encode(system_prompt))
    + doc_tokens
    + len(enc.encode(user_question))
)
output_tokens = 400  # 답변 토큰

INPUT_PRICE = 0.150 / 1_000_000
CACHED_PRICE = 0.075 / 1_000_000
OUTPUT_PRICE = 0.600 / 1_000_000

cost = input_tokens * INPUT_PRICE + output_tokens * OUTPUT_PRICE
cost_cached = input_tokens * CACHED_PRICE + output_tokens * OUTPUT_PRICE

print(f'input  tokens: {input_tokens:>8,}')
print(f'output tokens: {output_tokens:>8,}')
print(f'요청당 비용 (no cache)    : ${cost:.5f}')
print(f'요청당 비용 (cached input) : ${cost_cached:.5f}')
print()

# 1일 1000건 처리 시
print(f'일 1000건 비용 (no cache)   : ${cost * 1000:.2f}')
print(f'월 30,000건 비용 (no cache) : ${cost * 30_000:.2f}')
print(f'월 30,000건 비용 (cached)   : ${cost_cached * 30_000:.2f}')

In [ ]:
# 실제 사용량은 response_metadata의 token_usage에서 확인할 수 있다.
llm = get_llm(temperature=0.0)
resp = llm.invoke([
    SystemMessage(content=system_prompt),
    HumanMessage(content=user_question),
])
usage = resp.response_metadata.get('token_usage', {})
print('실제 usage:', usage)

# 실사용량 기반 비용
pt = usage.get('prompt_tokens', 0)
ct = usage.get('completion_tokens', 0)
real_cost = pt * INPUT_PRICE + ct * OUTPUT_PRICE
print(f'이번 호출 실제 비용: ${real_cost:.5f}')

## 과제

1. 위 tool calling 예제에 "현재 날짜를 알려주는 tool"을 추가하고, "어제 대비 오늘까지 90일간 이자"를 묻는 질의를 시도해 보세요.
2. `temperature=0.0`으로 실행한 모델도 100% 결정적이 아닙니다. 동일 프롬프트를 10번 실행해 유니크 응답 개수를 세어보세요.
3. 사내에서 하루 모델 호출 수를 가정하고 월 비용을 산출하세요. `cached input` 활용으로 줄일 수 있는 비율까지 기재해야 합니다.

## 더 읽어보기
- [OpenAI Pricing](https://openai.com/api/pricing/)
- [OpenAI Prompt Caching 가이드](https://platform.openai.com/docs/guides/prompt-caching)
- [LangChain Tool calling 가이드](https://python.langchain.com/docs/concepts/tool_calling/)
- [langchain-kr · 04-Model — 모델 사용법·스트리밍·비용](https://github.com/teddylee777/langchain-kr/tree/main/04-Model)